# LoRA sweep on Swahili dev (NER and POS)

Picks one LoRA setting per task before the baselines are retrained to convergence. The current LoRA (rank 8, attention
query and value only) was never tuned.

* **Configs:** rank 8, 16 and 32, crossed with `qv` (query and value, the current baseline) or `all` (every linear layer
  in each block). 6 configs per task.
* **Data:** Swahili only, seed 1. Up to 20 epochs; training stops early when the dev score has not improved for 3 epochs.
* **Test sets are never evaluated** (`--no_test`): the choice is made on dev only.
* **Selection rule, fixed in advance:** highest dev score; configs within 0.3 points count as tied, and the tie with the
  fewest trainable parameters wins (`src/sweep_report.py`).

**No Google Drive needed.** Code and data come from GitHub and the public datasets. Results are kept in `/content` and
downloaded as `sweep_results.zip` at the end. If Colab disconnects before that, the runs are lost and Run all starts
again. To keep partial results across disconnects, set `USE_DRIVE = True` in step 2.

**Before you start:** Runtime > Change runtime type > **L4 GPU**. Then Runtime > Run all.

**Time on an L4 (estimate):** NER 1 to 1.5 hours, POS 15 to 20 minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU, then run again."

## 2. Settings and where results go

In [ ]:
USE_DRIVE = False   # True: save results to Google Drive so a disconnect does not lose finished configs

from pathlib import Path
TASKS = ["ner", "pos"]
CONFIGS = [(r, t) for t in ["qv", "all"] for r in [8, 16, 32]]
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = Path("/content/drive/MyDrive/typology-aware-peft/results/sweep")
else:
    RESULTS = Path("/content/results/sweep")
RESULTS.mkdir(parents=True, exist_ok=True)
print("Results go to", RESULTS)

## 3. Get the code (latest version from GitHub)

In [ ]:
import os, shutil
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
if WORK.exists():
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO_URL} {WORK}
for name in ["src/token_baseline.py", "src/sweep_report.py", "src/sib200_report.py"]:
    assert (WORK / name).exists(), f"{name} is not on GitHub yet; push it first"
assert "--no_test" in (WORK / "src/token_baseline.py").read_text(), "token_baseline.py on GitHub is too old; push first"
os.chdir(WORK)
!git log --oneline -1

## 4. Install packages

Same pinned versions as the SIB-200 run. Colab now ships an old `torchao` that makes peft crash when it
builds a LoRA model (this stopped every LoRA run on 2 Nov), so it is removed; nothing here uses it.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
!pip uninstall -y -q torchao
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Smoke test (about 2 minutes)

One tiny run on the small base model with the new options (LoRA on all layers, early stopping, no test sets).

In [ ]:
for t in TASKS:
    !python src/token_baseline.py --task {t} --smoke --model Davlan/afro-xlmr-base --lora_targets all --lora_r 16 --no_test --out /content/smoke_{t}
    assert _exit_code == 0, f"smoke test for {t} failed; see the error above"

## 6. The sweep: POS first (quick), then NER

Each config writes to its own folder, so a rerun skips finished configs (only if `USE_DRIVE = True` survives a
disconnect). **Out of memory:** add `--bs 8` to the command and note it.

In [ ]:
for t in ["pos", "ner"]:
    for r, targets in CONFIGS:
        out = RESULTS / t / f"r{r}_{targets}"
        print(f"===== {t} rank {r} {targets} =====")
        !python src/token_baseline.py --task {t} --methods lora --langs swa --seeds 1 --epochs 20 --patience 3 --no_test --lora_r {r} --lora_targets {targets} --out {out}

## 7. Results and the chosen setting

In [ ]:
from IPython.display import Markdown, display
!python src/sweep_report.py --dir {RESULTS} > /dev/null
display(Markdown((RESULTS / "summary.md").read_text()))
print((RESULTS / "chosen.json").read_text())

## 8. Download the results

Leave `sweep_results.zip` in Downloads and tell Claude. It unzips into `results/sweep/` in the repo.

In [ ]:
from google.colab import files
shutil.make_archive("/content/sweep_results", "zip", RESULTS)
files.download("/content/sweep_results.zip")